# RT Anchor — quickstart

Calibrate an LC-MS lipidomics feature table to a portable, dimensionless **retention index (iRT)** anchored on a spiked standard panel. This notebook runs the bundled **Orbitrap** example (`input/samples.txt` + `input/standards.txt`).

## Install

```bash
pip install "rt-anchor[report]"
```

In [1]:
from rt_anchor import calibrate, write_results

res = calibrate(
    "input/samples.txt",                       # feature table to calibrate (format auto-detected)
    polarity="positive",
    standards_table="input/standards.txt",      # the standard-panel run (required)
)
res.model["calibration_scope"], res.model["n_anchors_used"]

('project', 13)

The original table is never altered — calibration only **appends** columns (`RI`, `RI_uncertainty`, `RI_reliability`, `RI_spread`, `is_extrapolated`, …):

In [2]:
cols = ["RI", "RI_uncertainty", "RI_reliability", "is_extrapolated"]
res.table[[res.rt_col, res.mz_col] + cols].head(8)

,RT,m/z,RI,RI_uncertainty,RI_reliability,is_extrapolated
0,7.394,758.5694,36.748923,0.731752,medium,False
1,8.434,765.6162,42.681180,0.458289,high,False
2,18.037,666.6185,97.713944,0.416613,high,False
3,8.762,786.6010,44.516892,0.452024,high,False
4,7.149,782.5693,35.563998,0.805704,medium,False
5,2.001,496.3394,NaN,NaN,none,True
6,7.394,759.5718,36.748923,0.731752,medium,False
7,17.791,874.7858,96.286812,0.416309,high,False


Model / QC summary:

In [3]:
m = res.model
{
    "scope": m["calibration_scope"],
    "anchors_used": m["n_anchors_used"],
    "anchor_span_min": m["anchor_span_min"],
    "loo_residual_irt_median": m["loo_residual_irt"]["median"],
    "n_features": m["n_features"],
}

{'scope': 'project',
 'anchors_used': 13,
 'anchor_span_min': [2.112, 18.42],
 'loo_residual_irt_median': 0.3537167678681925,
 'n_features': 22314}

## Write outputs

`write_results` emits the calibrated CSV, the fitted model (`_model.json`), the anchors, a log, and an interactive HTML + static PDF report into `output/`.

In [4]:
paths = write_results(res, "output/run")
sorted(paths)

['anchors_csv',
 'calibrated_csv',
 'log_txt',
 'model_json',
 'report_html',
 'report_pdf']

**Appended columns** — `RI` (iRT-style index), `RI_uncertainty` (heuristic, iRT units), `RI_reliability` (high/medium/low/none), `RI_spread` (per-sample dispersion), `is_extrapolated`, `calibration_scope`, `warp_source`.

See the rendered report at [`output/run_report.html`](output/run_report.html).